# PyMC in the browser, compiled

This notebook runs in [JupyterLite](https://jupyterlite.readthedocs.io/): Python is
Pyodide, in your browser, and no server is involved. PyMC installs and samples here
as it does anywhere, but PyTensor has no C compiler under Pyodide, so every log
density and gradient it evaluates runs on its Python linker, one node at a time.

[pymcwasm](https://github.com/habakan/pymcwasm) adds `mode="WASM"`: the graph is
compiled to a WebAssembly module by [tapewasm](https://github.com/habakan/tapewasm)
and run by the browser. There are two ways to use it here: keep PyMC's own NUTS and
change only what it calls, or hand the whole sampler to the module as well.

In [ ]:
import micropip
from pyodide.code import run_js

# The kernel runs in a worker under /extensions/, so the site root is the part of its
# URL before that; `location.origin` alone loses a project site's /repo/ prefix.
site_base = run_js('''(() => {
  const href = self.location.href, at = href.indexOf("/extensions/");
  return at === -1 ? self.location.origin : href.slice(0, at);
})()''')

# numba is a declared dependency of pytensor and pymc with no wasm build. Nothing
# here calls it, so the two install without their declared dependencies, and what
# they do import is listed; setuptools is what PyTensor's VM imports to check for C.
for p in ["setuptools", "filelock", "cachetools", "typing-extensions", "cloudpickle", "cons",
          "logical-unification", "etuples", "miniKanren", "rich", "threadpoolctl",
          "xarray", "arviz", "scipy", "pandas"]:
    try:
        await micropip.install(p)
    except Exception as e:
        print("skipped", p, type(e).__name__)
await micropip.install(["pytensor", "pymc"], deps=False)
await micropip.install(site_base + "/files/pymcwasm-0.1.0-py3-none-any.whl", deps=False)

import warnings
warnings.filterwarnings("ignore")   # a Pyodide deprecation warning from threadpoolctl

import pytensor
pytensor.config.mode = "FAST_COMPILE"
pytensor.config.linker = "py"   # what PyTensor falls back to here anyway

import logging
import numpy as np, pymc as pm, arviz as az
logging.getLogger("pymc").setLevel(logging.ERROR)   # after the import, which sets it to INFO
import pymcwasm.linker
await pymcwasm.linker.load(site_base + "/files/tapewasm/pkg/tapewasm.js")
print("pymc", pm.__version__, "pytensor", pytensor.__version__)

## A model

Eight schools, non-centred: each school's coaching effect, partially pooled.

In [ ]:
y = np.array([28, 8, -3, 7, -1, 1, 18, 12], dtype=float)
sigma = np.array([15, 10, 16, 11, 9, 11, 10, 18], dtype=float)

with pm.Model() as model:
    mu = pm.Normal("mu", 0, 5)
    tau = pm.HalfCauchy("tau", 5)
    z = pm.Normal("z", 0, 1, shape=8)
    theta = pm.Deterministic("theta", mu + tau * z)
    pm.Normal("y", theta, sigma, observed=y)

## The same sampler, two ways to evaluate the density

`compile_kwargs={"mode": "WASM"}` is the whole change.

In [ ]:
import time

def timed(**kw):
    with model:
        t = time.perf_counter()
        idata = pm.sample(500, tune=500, chains=2, cores=1, random_seed=1,
                          progressbar=False, **kw)
    return time.perf_counter() - t, idata

python_s, idata_python = timed()
wasm_s, idata = timed(compile_kwargs={"mode": "WASM"})
print(f"Python linker {python_s:.1f} s, WASM {wasm_s:.1f} s: {python_s / wasm_s:.1f}x")

In [ ]:
az.summary(idata, var_names=["mu", "tau"])

## The sampler in the module too

Most of what is left is PyMC's NUTS itself, in Python. `pymcwasm.sample` compiles the
same log density once and draws with [nuts-rs](https://github.com/pymc-devs/nuts-rs)
inside the module, so no Python runs per step. Compiling is the part that costs; the
draws are what repeats.

In [ ]:
import pymcwasm

t = time.perf_counter()
compiled = await pymcwasm.compile(model, tapewasm_path=site_base + "/files/tapewasm/pkg/tapewasm.js")
compile_s = time.perf_counter() - t
t = time.perf_counter()
fit = await compiled.sample(draws=500, warmup=500, chains=2, seed=1)
draw_s = time.perf_counter() - t
print(f"compile {compile_s:.2f} s, draw {draw_s:.2f} s")
print(f"PyMC's NUTS: Python linker {python_s:.1f} s, WASM {wasm_s:.1f} s; nuts-rs in the module {draw_s:.2f} s")
az.summary(fit.to_inference_data(), var_names=["mu", "tau"])

## Is it the same density?

Both modes evaluate PyMC's own log density and gradient; at a point away from the
start they agree to rounding.

In [ ]:
point = {k: v + 0.3 for k, v in model.initial_point().items()}
for name in ("logp", "dlogp"):
    f = getattr(model, "compile_" + name)
    a, b = np.asarray(f()(point)), np.asarray(f(mode="WASM")(point))
    print(name, "max relative difference", float(np.max(np.abs(a - b) / np.maximum(1, np.abs(a)))))